# Workshop Sequential: Triaje → Plan Clínico (Hospital)

## 🎯 ¿Qué es el Patrón Sequential?

El patrón **Sequential** es una estrategia de orquestación donde los agentes trabajan en una **cadena fija de pasos**.
La salida del primer agente se utiliza como entrada del siguiente.

Es ideal cuando quieres un flujo tipo *pipeline*: primero **estructurar** información, después **decidir/actuar**.


## 🏥 Escenario del ejercicio (muy concreto)

En Urgencias llega un paciente y quieres simular un flujo realista de dos fases (triaje → plan):
> “Paciente de 67 años con dolor torácico opresivo y disnea. Antecedentes: HTA y DM2. TA 90/60, SatO2 92%.”

Lo que buscamos demostrar con **Sequential** es esto:
1. `EnfermeriaTriaje` estructura datos, detecta señales de alarma y prioriza.
2. `MedicoUrgencias` convierte ese triaje en un plan clínico inicial accionable.

### ✅ Qué deberías ver en la salida
- Un output de `EnfermeriaTriaje` con datos que faltan, riesgos y prioridad (alta/media/baja).
- Un output de `MedicoUrgencias` con plan inicial breve y seguro.
- La conversación completa en orden: usuario → triaje → médico.

---

## Paso 0: Importaciones y configuración

In [1]:
import os
import time
from agent_framework import ChatAgent, SequentialBuilder, WorkflowOutputEvent
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
print(AZURE_OPENAI_DEPLOYMENT)

print("✅ Entorno cargado y Microsoft Agent Framework importado")

gpt-5.4
✅ Entorno cargado y Microsoft Agent Framework importado


## Paso 1: Crear dos agentes con roles especializados
- **EnfermeriaTriaje**: recoge datos, prioriza y detecta riesgos
- **MedicoUrgencias**: convierte el triaje en un plan de actuación

In [2]:
# Agente 1: Enfermería de triaje
analista = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_DEPLOYMENT
    ),
    name="EnfermeriaTriaje",
    instructions="""Eres una enfermera de triaje en Urgencias.
Tu tarea es estructurar la información del caso y priorizar.

Incluye:
- Datos clave que faltan (preguntas concretas)
- Señales de alarma / riesgos inmediatos
- Prioridad sugerida (alta/media/baja) y por qué

Responde en español. Sé concisa y práctica."""
 )

# Agente 2: Médico de urgencias
redactor = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_DEPLOYMENT
    ),
    name="MedicoUrgencias",
    instructions="""Eres médico de urgencias.
Con la evaluación de triaje, propone un plan inicial claro.

Incluye:
- Hipótesis principales (sin diagnóstico definitivo)
- Pruebas/acciones iniciales prioritarias
- Riesgos a vigilar y criterios de escalado

Responde en español. Sé claro y accionable."""
 )

print("✅ Dos agentes clínicos creados")

✅ Dos agentes clínicos creados


## Paso 2: Crear workflow secuencial con SequentialBuilder (API oficial de Microsoft Agent Framework)
Con `SequentialBuilder`, el framework gestiona automáticamente:
- Ejecución secuencial (agente 1 → agente 2)
- Pasaje del contexto completo entre agentes
- Agregación de resultados

In [3]:
# PATRÓN OFICIAL DE MICROSOFT AGENT FRAMEWORK
# SequentialBuilder crea un pipeline donde:
# - Agente 1 recibe: prompt inicial
# - Agente 2 recibe: conversación completa de agente 1 + su output
# - Agente N recibe: toda la conversación anterior

workflow_secuencial = SequentialBuilder().participants([analista, redactor]).build()

print("✅ Workflow secuencial creado con SequentialBuilder")

✅ Workflow secuencial creado con SequentialBuilder


## Paso 3: Ejecutar el workflow secuencial
El workflow ejecuta automáticamente: EnfermeriaTriaje → MedicoUrgencias

In [4]:
async def flujo_colaborativo():
    tema = (
        "Paciente de 67 años con dolor torácico opresivo y disnea. "
        "Antecedentes: HTA y DM2. TA 90/60, SatO2 92%. "
        "¿Cómo lo priorizas y qué plan inicial propones?"
    )
    
    print(f"\n{'='*80}")
    print("FLUJO SECUENCIAL CON SequentialBuilder (Microsoft Agent Framework)")
    print(f"{'='*80}")
    print(f"📊 Caso: {tema}\n")
    print("⏳ Ejecutando workflow secuencial (EnfermeriaTriaje → MedicoUrgencias)...\n")
    
    inicio = time.time()
    
    # Ejecutar workflow y capturar eventos
    output_evt = None
    async for event in workflow_secuencial.run_stream(tema):
        if isinstance(event, WorkflowOutputEvent):
            output_evt = event
            break
    
    tiempo_total = time.time() - inicio
    
    print(f"✅ Workflow completado en {tiempo_total:.2f}s\n")
    
    if output_evt:
        messages = output_evt.data
        
        print(f"{'='*80}")
        print("CONVERSACIÓN COMPLETA (todas las fases)")
        print(f"{'='*80}\n")
        
        # Mostrar todos los mensajes del workflow
        for i, msg in enumerate(messages, 1): # type: ignore
            author = msg.author_name or ("usuario" if msg.role.value == "user" else "asistente")
            print(f"{i}. [{author}]:")
            print("-" * 80)
            content = msg.text if hasattr(msg, 'text') else str(msg.content)
            print(f"{content}\n")

# Ejecutar el flujo colaborativo
resultado = await flujo_colaborativo()


FLUJO SECUENCIAL CON SequentialBuilder (Microsoft Agent Framework)
📊 Caso: Paciente de 67 años con dolor torácico opresivo y disnea. Antecedentes: HTA y DM2. TA 90/60, SatO2 92%. ¿Cómo lo priorizas y qué plan inicial propones?

⏳ Ejecutando workflow secuencial (EnfermeriaTriaje → MedicoUrgencias)...

✅ Workflow completado en 30.99s

CONVERSACIÓN COMPLETA (todas las fases)

1. [usuario]:
--------------------------------------------------------------------------------
Paciente de 67 años con dolor torácico opresivo y disnea. Antecedentes: HTA y DM2. TA 90/60, SatO2 92%. ¿Cómo lo priorizas y qué plan inicial propones?

2. [EnfermeriaTriaje]:
--------------------------------------------------------------------------------
**Prioridad: ALTA / atención inmediata (emergencia).**

**Por qué**
- **Dolor torácico opresivo + disnea** en paciente de **67 años** con **HTA y DM2**.
- **TA 90/60**: posible **inestabilidad hemodinámica**.
- **SatO2 92%**: hipoxemia leve.
- Riesgo de **síndrome corona